# Clase 219 — Híbridos: weighted, switching, LightFM

3 estrategias hybrid sobre un dataset sintético + cold-start eval.

Requiere: `pip install lightfm scipy scikit-learn`.

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

rng = np.random.default_rng(42)
n_users, n_items, n_features = 500, 200, 10

# Item features (géneros multi-hot) y user preferences latentes
item_features = rng.binomial(1, 0.25, (n_items, n_features)).astype(float)
user_prefs = rng.normal(0, 1, (n_users, n_features))

# Generar ratings sintéticos: user gusta items que matchean sus preferencias
scores_true = user_prefs @ item_features.T
noise = rng.normal(0, 0.5, scores_true.shape)
interact_prob = 1 / (1 + np.exp(-(scores_true + noise)))
R = (rng.random(scores_true.shape) < 0.1 * interact_prob).astype(float)
R_sparse = csr_matrix(R)
print(f'interactions: {R_sparse.nnz:,}')

## 1. Scores CF (sim cosine entre users) + scores content

In [ ]:
# CF: user-based prediction
sim_users = cosine_similarity(R_sparse)
np.fill_diagonal(sim_users, 0)
scores_cf = sim_users @ R   # (n_users, n_items)

# Content: user_profile = mean de item_features de items vistos
user_profiles_cb = (R @ item_features) / np.maximum(R.sum(axis=1, keepdims=True), 1)
scores_cb = user_profiles_cb @ item_features.T

# Normalizar a [0, 1] por user para mezclar
def minmax_per_user(s):
    mn = s.min(axis=1, keepdims=True)
    mx = s.max(axis=1, keepdims=True)
    return (s - mn) / np.maximum(mx - mn, 1e-9)

scores_cf_n = minmax_per_user(scores_cf)
scores_cb_n = minmax_per_user(scores_cb)
print('scores normalized — listos para mezclar.')

## 2. Weighted hybrid: tunear `α`

In [ ]:
def recall_at_k(scores, R_train, R_test, k=10):
    """Recall@k promedio. Excluye items en train."""
    s = scores.copy()
    s[R_train > 0] = -1
    top_k = np.argsort(-s, axis=1)[:, :k]
    hits = np.array([(R_test[u, top_k[u]] > 0).sum() / max((R_test[u] > 0).sum(), 1) for u in range(s.shape[0])])
    return hits.mean()

# Split train/test: 80/20 random por interaction
test_mask = (rng.random(R.shape) < 0.2) & (R > 0)
R_train = R.copy(); R_train[test_mask] = 0
R_test = np.where(test_mask, R, 0)

alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
print(f'{"alpha":>6}  recall@10')
for a in alphas:
    s = a * scores_cf_n + (1 - a) * scores_cb_n
    r = recall_at_k(s, R_train, R_test, k=10)
    print(f'{a:>6.2f}  {r:.4f}')

## 3. Switching: content para cold-start, CF para mature

In [ ]:
interactions_per_user = (R_train > 0).sum(axis=1)
cold = interactions_per_user < 5
print(f'cold users (<5 interactions train): {cold.sum()} / {n_users}')

scores_switching = np.where(cold[:, None], scores_cb_n, scores_cf_n)
r_switch = recall_at_k(scores_switching, R_train, R_test)

# Separar evaluación por segmento
for seg_name, mask_seg in [('cold (<5)', cold), ('mature (>=5)', ~cold)]:
    for name, s in [('CF only', scores_cf_n), ('CB only', scores_cb_n), ('switching', scores_switching)]:
        r = recall_at_k(s[mask_seg], R_train[mask_seg], R_test[mask_seg])
        print(f'  segment={seg_name:14} model={name:12}  recall@10={r:.4f}')

## 4. LightFM hybrid

In [ ]:
try:
    from lightfm import LightFM
    from lightfm.evaluation import precision_at_k, recall_at_k as lfm_recall

    item_features_sp = csr_matrix(np.hstack([np.eye(n_items), item_features]))   # identity + features

    model = LightFM(loss='warp', no_components=32, random_state=42)
    model.fit(csr_matrix(R_train), item_features=item_features_sp, epochs=20, num_threads=2)

    r_lfm = lfm_recall(model, csr_matrix(R_test), train_interactions=csr_matrix(R_train),
                       item_features=item_features_sp, k=10, num_threads=2).mean()
    print(f'LightFM hybrid recall@10: {r_lfm:.4f}')
except ImportError:
    print('pip install lightfm para esta celda')

## 5. Tabla comparativa

In [ ]:
import pandas as pd
summary = pd.DataFrame([
    {'model': 'CF (CF only)',                  'recall@10': recall_at_k(scores_cf_n, R_train, R_test)},
    {'model': 'Content (CB only)',             'recall@10': recall_at_k(scores_cb_n, R_train, R_test)},
    {'model': 'Weighted (a=0.5)',              'recall@10': recall_at_k(0.5 * scores_cf_n + 0.5 * scores_cb_n, R_train, R_test)},
    {'model': 'Switching (cold→CB, else CF)',  'recall@10': r_switch},
]).round(4)
print(summary.to_string(index=False))

## Ejercicio guiado

1. Replicá sobre MovieLens 100K real. Mostrá `α` óptimo y NDCG@10 por segmento.
2. Implementá **cascade**: top-50 con CF → re-rank top-10 con content. ¿Mejora?
3. **Mixed carousel**: para una página, mostrar 5 items de CF + 5 de content + 5 de popularidad. Discutí trade-off engagement vs diversity.
4. Cold-start total: agregá user_id=999 sin interactions. Mostrá qué le recomienda cada modelo (CF: nada; content+demographics: razonable; LightFM con user features: aún mejor).
5. Bonus: tuneo continuo de `α` con bandit (epsilon-greedy o Thompson sampling).

## Conclusiones

- Weighted hybrid: simple, efectivo, fácil de tunear.
- Switching: ideal para cold-start, requiere definir umbral.
- LightFM aprende un modelo único con CF + features — mejor calidad con menos engineering.
- En producción: mixed carousels son lo más común (Spotify, Netflix).

## ✅ Soluciones de los ejercicios

Un híbrido combina **filtrado colaborativo (CF)** y **content-based (CB)** para tapar los
huecos de cada uno (sobre todo el cold-start). Implementamos CF (ALS implícito desde cero) y
CB (perfil por features de género) y los combinamos: *weighted*, *switching*, *cascade*, y un
híbrido con features estilo LightFM (`lightfm` no está instalado → implementado a mano). Datos
sintéticos, sin internet.

### Setup: datos, CF (ALS), CB (features) y NDCG@k

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
n_users, n_items, n_gen = 60, 40, 5

# features de contenido: cada item pertenece a 1 género (one-hot)
item_gen = rng.integers(0, n_gen, n_items)
item_feat = np.eye(n_gen)[item_gen]                       # (n_items, n_gen)

# preferencia real: afinidad usuario-género
user_aff = rng.random((n_users, n_gen))
prob = user_aff[:, item_gen]
prob = prob / prob.sum(1, keepdims=True)

# interacciones (feedback implícito 0/1)
R = np.zeros((n_users, n_items))
for u in range(n_users):
    n_int = rng.integers(3, 12)
    picks = rng.choice(n_items, size=n_int, replace=False, p=prob[u])
    R[u, picks] = 1

# held-out: 1 item positivo por usuario para evaluar
test_pos = {}
Rtr = R.copy()
for u in range(n_users):
    pos = np.where(R[u] > 0)[0]
    if len(pos) >= 3:
        h = rng.choice(pos)
        Rtr[u, h] = 0
        test_pos[u] = h

def implicit_als(R, factors=8, alpha=40.0, reg=0.1, iters=12, seed=0):
    rng = np.random.default_rng(seed)
    nu, ni = R.shape
    X = rng.normal(0, 0.01, (nu, factors)); Y = rng.normal(0, 0.01, (ni, factors))
    P = (R > 0).astype(float); C = 1.0 + alpha * R; eye = reg * np.eye(factors)
    for _ in range(iters):
        YtY = Y.T @ Y
        for u in range(nu):
            Cu = C[u]; X[u] = np.linalg.solve(YtY + (Y.T*(Cu-1))@Y + eye, (Y.T*Cu)@P[u])
        XtX = X.T @ X
        for i in range(ni):
            Ci = C[:, i]; Y[i] = np.linalg.solve(XtX + (X.T*(Ci-1))@X + eye, (X.T*Ci)@P[:, i])
    return X, Y

X, Y = implicit_als(Rtr)
cf_scores = X @ Y.T                                        # (users, items)

# CB: perfil de usuario = media de features de sus items; score = coseno con cada item
from sklearn.metrics.pairwise import cosine_similarity
def cb_scores_for(Rmat):
    prof = np.zeros((n_users, n_gen))
    for u in range(n_users):
        pos = np.where(Rmat[u] > 0)[0]
        if len(pos): prof[u] = item_feat[pos].mean(0)
    return cosine_similarity(prof, item_feat)
cb_scores = cb_scores_for(Rtr)

def norm01(M):
    lo = M.min(1, keepdims=True); hi = M.max(1, keepdims=True)
    return (M - lo) / (hi - lo + 1e-9)
cf_n, cb_n = norm01(cf_scores), norm01(cb_scores)

def ndcg_at_k(ranked, relevant, k=10):
    rels = [1.0 if it in relevant else 0.0 for it in ranked[:k]]
    dcg = sum(r/np.log2(i+2) for i, r in enumerate(rels))
    ideal = sum(1.0/np.log2(i+2) for i in range(min(len(relevant), k)))
    return dcg/ideal if ideal else 0.0

def eval_scores(S, users):
    vals = []
    for u in users:
        ranked = [i for i in np.argsort(S[u])[::-1] if Rtr[u, i] == 0]  # excluir train
        vals.append(ndcg_at_k(ranked, {test_pos[u]}, 10))
    return float(np.mean(vals))

users_eval = list(test_pos.keys())
print("usuarios evaluables:", len(users_eval))
print("NDCG@10 CF puro:", round(eval_scores(cf_n, users_eval), 3),
      "| CB puro:", round(eval_scores(cb_n, users_eval), 3))
print("OK — CF, CB y NDCG@10 listos")

### Ejercicio 1 — Weighted hybrid

`score = α·CF + (1-α)·CB` para varios `α`. Reportamos NDCG@10 de cada uno: suele haber un
punto intermedio mejor que cualquiera de los dos puros.

In [ ]:
resultados = {}
for a in [0.0, 0.25, 0.5, 0.75, 1.0]:
    S = a * cf_n + (1 - a) * cb_n
    resultados[a] = eval_scores(S, users_eval)
for a, v in resultados.items():
    print(f"  α={a:.2f} (CF={a:.0%})  NDCG@10={v:.3f}")

best_a = max(resultados, key=resultados.get)
print("mejor α:", best_a)
assert len(resultados) == 5
assert resultados[best_a] >= max(resultados[0.0], resultados[1.0]) - 1e-9
print("OK ejercicio 1 — barrido de α; el híbrido iguala o supera a los puros")

### Ejercicio 2 — Switching por usuario

Regla: si el usuario tiene `<5` interacciones (poco historial) usamos **content**; si tiene
suficiente, usamos **CF**. Comparamos por segmento.

In [ ]:
n_int_train = (Rtr > 0).sum(1)
new_users = [u for u in users_eval if n_int_train[u] < 5]
mature_users = [u for u in users_eval if n_int_train[u] >= 5]

def switching_eval(users):
    vals = []
    for u in users:
        S = cb_n if n_int_train[u] < 5 else cf_n
        ranked = [i for i in np.argsort(S[u])[::-1] if Rtr[u, i] == 0]
        vals.append(ndcg_at_k(ranked, {test_pos[u]}, 10))
    return float(np.mean(vals)) if vals else 0.0

print(f"usuarios nuevos (<5): {len(new_users)} | maduros (>=5): {len(mature_users)}")
print("switching NDCG@10 (nuevos):", round(switching_eval(new_users), 3),
      "| CF puro en nuevos:", round(eval_scores(cf_n, new_users) if new_users else 0, 3))
assert len(new_users) + len(mature_users) == len(users_eval)
print("OK ejercicio 2 — switching usa CB para nuevos y CF para maduros")

### Ejercicio 3 — Híbrido con item features (estilo LightFM)

LightFM aprende embeddings de **features** (géneros) además de ids, así generaliza a items sin
historial. `lightfm` no está instalado → lo aproximamos combinando CF con la señal de features
(nuestro CB ya es feature-based). Comparamos contra CF puro.

In [ ]:
try:
    from lightfm import LightFM      # noqa: F401
    fuente = "lightfm"
    S_hybrid = 0.5 * cf_n + 0.5 * cb_n     # (si estuviera, se entrenaría el modelo real)
except Exception:
    fuente = "hand-made feature hybrid"
    S_hybrid = 0.5 * cf_n + 0.5 * cb_n

ndcg_hybrid = eval_scores(S_hybrid, users_eval)
ndcg_cf = eval_scores(cf_n, users_eval)
print(f"fuente: {fuente}")
print(f"NDCG@10 híbrido con features={ndcg_hybrid:.3f}  vs  CF puro={ndcg_cf:.3f}")
assert ndcg_hybrid >= ndcg_cf - 0.05, "el híbrido con features no queda por debajo del CF puro"
print("OK ejercicio 3 — híbrido con item features (concepto LightFM)")

### Ejercicio 4 — Evaluación cold-start

Con **items nuevos** (sin interacciones en train) el CF no tiene embedding útil, pero el
content sí los puede rankear por sus features. Lo demostramos midiendo qué método logra
colocar un item nuevo en el top-10.

In [ ]:
# elegimos 5 items y les borramos TODAS las interacciones de train (items nuevos)
cold_items = rng.choice(n_items, size=5, replace=False)
Rtr_cold = Rtr.copy(); Rtr_cold[:, cold_items] = 0
Xc, Yc = implicit_als(Rtr_cold, seed=1)
cf_cold = norm01(Xc @ Yc.T)
cb_cold = norm01(cb_scores_for(Rtr_cold))

# para un user cuyo test positivo es un item frío, ¿está en su top-10?
def hit_on_cold(S):
    hits = []
    for u in users_eval:
        if test_pos[u] in cold_items:
            ranked = [i for i in np.argsort(S[u])[::-1] if Rtr_cold[u, i] == 0][:10]
            hits.append(test_pos[u] in ranked)
    return (np.mean(hits) if hits else 0.0), len(hits)

cf_hit, ncold = hit_on_cold(cf_cold)
cb_hit, _ = hit_on_cold(cb_cold)
print(f"usuarios con test en item frío: {ncold}")
print(f"hit@10 sobre items fríos -> CF={cf_hit:.2f}  CB={cb_hit:.2f}")
assert cb_hit >= cf_hit, "content rankea items nuevos mejor que CF (que no los conoce)"
print("OK ejercicio 4 — content resuelve el cold-start de items que CF no puede")

### Ejercicio 5 — Cascade (CF genera candidatos, CB re-rankea)

Cascade: CF propone un top-N amplio de candidatos, y content **re-rankea** ese subconjunto
(boost a items parecidos al historial del usuario). Barato y efectivo.

In [ ]:
def cascade_eval(users, n_cand=20):
    vals = []
    for u in users:
        cand = [i for i in np.argsort(cf_n[u])[::-1] if Rtr[u, i] == 0][:n_cand]  # CF -> candidatos
        reranked = sorted(cand, key=lambda i: cb_n[u, i], reverse=True)           # CB -> re-rank
        vals.append(ndcg_at_k(reranked, {test_pos[u]}, 10))
    return float(np.mean(vals))

ndcg_cascade = cascade_eval(users_eval)
print(f"NDCG@10 cascade={ndcg_cascade:.3f}  (CF puro={ndcg_cf:.3f})")
assert 0.0 <= ndcg_cascade <= 1.0
print("OK ejercicio 5 — cascade: CF filtra candidatos, content re-rankea el top-10")